In [ ]:
import time
import math
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import init
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.metrics import f1_score
from sklearn.model_selection import KFold
from pathlib import Path
from torchinfo import summary
from brainvision.constants import *


In [45]:
# ══════════════════════════════════════════════════════════════════════════════
#                        CONFIGURE EXPERIMENT HERE
# ══════════════════════════════════════════════════════════════════════════════

MODEL    = "1D-NN"    # "1D-NN" | "1D-CNN" | "2D-CNN" | "3D-CNN" | "SF"
LOSS_FN  = "CE"       # "CE" | "FL" | "DL" | "UFL"
STRATEGY = "vp1"      # "vp1" | "vp2" | "vp3"
N_FOLDS  = 5          # only used when STRATEGY = "vp3"

In [ ]:
from brainvision.data import load_processed_patients

def load_all_campaigns() -> dict[int, list[dict]]:
    """Load preprocessed patients for all three campaigns."""
    print("Loading all campaigns...")
    return {
        c: load_processed_patients(PROCESSED_DIRS[c])
        for c in [1, 2, 3]
    }

In [ ]:
from brainvision.validation import build_splits_vp1, build_splits_vp2, build_splits_vp3, verify_no_leakage

campaigns = load_all_campaigns()

if STRATEGY == 'vp1':
    splits = build_splits_vp1(campaigns)
    verify_no_leakage(splits)
    all_splits = [{'fold': None, **splits}]    # wrap as single fold for uniform training loop

elif STRATEGY == 'vp2':
    splits = build_splits_vp2(campaigns)
    verify_no_leakage(splits)
    all_splits = [{'fold': None, **splits}]

elif STRATEGY == 'vp3':
    all_splits = build_splits_vp3(campaigns, n_folds=N_FOLDS)
    for fold in all_splits:
        verify_no_leakage(fold)

else:
    raise ValueError(f"Unknown STRATEGY '{STRATEGY}'. Choose 'vp1', 'vp2', or 'vp3'.")

Loading all campaigns...
  Loaded  27 patients from /kaggle/input/datasets/odelolajoshua/brain-hsi-processed/first_campaign
  Loaded  24 patients from /kaggle/input/datasets/odelolajoshua/brain-hsi-processed/second_campaign
  Loaded  10 patients from /kaggle/input/datasets/odelolajoshua/brain-hsi-processed/third_campaign

VP1 splits:
  train:  35 images  21 patients → ['004', '005', '007', '008', '010', '012', '013', '014', '015', '017', '018', '020', '021', '022', '035', '036', '038', '039', '040', '041', '042']
  val  :  16 images   5 patients → ['016', '019', '034', '037', '043']
  test :  10 images   8 patients → ['050', '051', '053', '054', '055', '056', '057', '058']
✅ No patient-level leakage detected


In [ ]:
from brainvision.losses import DiceLoss, FocalLoss, UnifiedFocalLoss
from brainvision.models import Baseline1DDNN, HuEtAl1DCNN, LeeEtAl2DCNN, HamidaEtAl3DCNN, HybridSN, SpectralFormer

MODEL_REGISTRY = {
    '1D-NN': lambda: Baseline1DDNN(
        input_channels  = N_DECIMATED_BANDS,
        n_classes       = N_CLASSES,
        dropout         = True,
        dropout_rate    = DROPOUT_RATE
    ),

    '1D-CNN': lambda: HuEtAl1DCNN(
        input_channels = N_DECIMATED_BANDS,
        n_classes      = N_CLASSES
    ),

    '2D-CNN': lambda: LeeEtAl2DCNN(
        input_channels = N_DECIMATED_BANDS,
        n_classes      = N_CLASSES,
        patch_size     = PATCH_SIZE
    ),

    '3D-CNN': lambda: HamidaEtAl3DCNN(
        input_channels = N_DECIMATED_BANDS,
        n_classes      = N_CLASSES,
        patch_size     = PATCH_SIZE
    ),
    
    'HybridSN': lambda: HybridSN(
        input_channels = N_DECIMATED_BANDS,
        n_classes      = N_CLASSES,
        patch_size     = PATCH_SIZE
    ),
    
    'SpectralFormer': lambda: SpectralFormer(
        input_channels = N_DECIMATED_BANDS,
        n_classes      = N_CLASSES,
        near_band      = SF_NEAR_BAND,
        dim            = SF_DIM,
        depth          = SF_DEPTH,
        heads          = SF_HEADS,
        dim_head       = SF_DIM_HEAD,
        mlp_dim        = SF_MLP_DIM,
        dropout        = SF_DROPOUT,
        emb_dropout    = SF_EMB_DROPOUT,
        mode           = SF_MODE,
    ),
}

PATCH_MODELS = {'2D-CNN', '3D-CNN', 'HybridSN'}

LOSS_REGISTRY = {
    'CE'  : lambda w: nn.CrossEntropyLoss(weight=w),
    'FL'  : lambda w: FocalLoss(alpha=w),
    'DL'  : lambda w: DiceLoss(),
    'UFL' : lambda w: UnifiedFocalLoss(alpha=w),
}

In [66]:
def model_summary(model_name: str):
    """
    Print torchinfo summary for a given model.
    Uses the correct input shape based on whether it's
    a pixel model or a patch model.
    """
    model = MODEL_REGISTRY[model_name]()

    if model_name in PATCH_MODELS:
        input_size = (1, N_DECIMATED_BANDS, PATCH_SIZE, PATCH_SIZE)
    else:
        input_size = (1, N_DECIMATED_BANDS)

    print(f"\n{'═'*60}")
    print(f"  {model_name}")
    print(f"{'═'*60}")

    result = summary(
        model,
        input_size   = input_size,
        col_names    = ['num_params', 'kernel_size', 'mult_adds',
                        'input_size', 'output_size'],
        col_width    = 12,
        row_settings = ['var_names'],
        depth        = 4,
        device       = 'cpu',
        verbose      = 0,          # ← suppress auto-print
    )

    print(result)                  # ← explicit print forces display in Jupyter

In [67]:
device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
x_pixel = torch.randn(4, N_DECIMATED_BANDS).to(device)
x_patch = torch.randn(4, N_DECIMATED_BANDS, PATCH_SIZE, PATCH_SIZE).to(device)

print("── Output shape checks ───────────────────────────────")
for name, builder in MODEL_REGISTRY.items():
    m   = builder().to(device)
    x   = x_patch if name in PATCH_MODELS else x_pixel
    out = m(x)
    print(f"  {name:<16} input={tuple(x.shape)}  "
          f"output={tuple(out.shape)}  "
          f"params={sum(p.numel() for p in m.parameters()):,}")

print("\n── Detailed summaries ────────────────────────────────")
for name in MODEL_REGISTRY:
    model_summary(name)

── Output shape checks ───────────────────────────────
  1D-NN            input=(4, 128)  output=(4, 4)  params=17,055,748
  1D-CNN           input=(4, 128)  output=(4, 4)  params=76,824
  2D-CNN           input=(4, 128, 5, 5)  output=(4, 4)  params=296,580
  3D-CNN           input=(4, 128, 5, 5)  output=(4, 4)  params=33,004
  HybridSN         input=(4, 128, 5, 5)  output=(4, 4)  params=2,601,588
  SpectralFormer   input=(4, 128)  output=(4, 4)  params=198,197

── Detailed summaries ────────────────────────────────

════════════════════════════════════════════════════════════
  1D-NN
════════════════════════════════════════════════════════════
Layer (type (var_name))                  Param #      Kernel Shape Mult-Adds    Input Shape  Output Shape
Baseline1DDNN (Baseline1DDNN)            --           --           --           [1, 128]     [1, 4]
├─Linear (fc1)                           264,192      --           264,192      [1, 128]     [1, 2048]
├─Dropout (dropout)                   

In [70]:
def compute_class_weights(dataset: Dataset) -> torch.Tensor:
    counts  = torch.bincount(dataset.y, minlength=N_CLASSES).float()
    weights = 1.0 / (counts + 1e-6)
    weights = weights / weights.sum()
    print("\nClass weights (from training set):")
    for i, (c, w) in enumerate(zip(counts, weights)):
        print(f"  {CLASS_NAMES[i+1]:<25}: {int(c):>10,} px  →  {w:.4f}")
    return weights

In [ ]:
from brainvision.data import HSIPatchDataset, HSIPixelDataset


def build_loaders(train_patients, val_patients):
    """Build train and val DataLoaders based on MODEL."""
    if MODEL in PATCH_MODELS:
        tr_ds = HSIPatchDataset(train_patients)
        va_ds = HSIPatchDataset(val_patients)
    else:
        tr_ds = HSIPixelDataset(train_patients)
        va_ds = HSIPixelDataset(val_patients)

    tr_loader = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=0, pin_memory=True)
    va_loader = DataLoader(va_ds, batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=0, pin_memory=True)
    weights   = compute_class_weights(tr_ds)
    return tr_loader, va_loader, weights

## Training

In [72]:
def train_one_epoch(model, loader, optimiser, criterion, device):
    model.train()
    total_loss, n = 0.0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        optimiser.zero_grad()
        loss = criterion(model(X), y)
        loss.backward()
        optimiser.step()
        total_loss += loss.item() * len(y)
        n          += len(y)
    return total_loss / n

In [73]:
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, n          = 0.0, 0
    all_preds, all_targets = [], []

    for X, y in loader:
        X, y   = X.to(device), y.to(device)
        logits = model(X)
        total_loss  += criterion(logits, y).item() * len(y)
        n           += len(y)
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_targets.extend(y.cpu().numpy())

    preds   = np.array(all_preds)
    targets = np.array(all_targets)

    macro_f1    = f1_score(targets, preds, average='macro', zero_division=0)
    sensitivity, specificity = _per_class_sens_spec(targets, preds)

    return (total_loss / n,
            macro_f1,
            sensitivity.mean(),    # macro-averaged sensitivity
            specificity.mean())    # macro-averaged specificity


def _per_class_sens_spec(targets: np.ndarray,
                          preds:   np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """
    Compute per-class sensitivity (recall) and specificity from
    a confusion matrix. Returns arrays of shape (N_CLASSES,).
    """
    sensitivity = np.zeros(N_CLASSES)
    specificity = np.zeros(N_CLASSES)

    for i in range(N_CLASSES):
        TP = ((preds == i) & (targets == i)).sum()
        FN = ((preds != i) & (targets == i)).sum()
        FP = ((preds == i) & (targets != i)).sum()
        TN = ((preds != i) & (targets != i)).sum()

        sensitivity[i] = TP / (TP + FN + 1e-6)
        specificity[i] = TN / (TN + FP + 1e-6)

    return sensitivity, specificity

In [74]:
def train(model, train_loader, val_loader, criterion, device,
          checkpoint_path: str) -> dict:

    optimiser = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    scheduler = ReduceLROnPlateau(optimiser, mode='max',
                                  factor=LR_DECAY_FACTOR,
                                  patience=LR_DECAY_PATIENCE)

    history    = {
        'train_loss'  : [],
        'val_loss'    : [],
        'val_f1'      : [],
        'val_sens'    : [],    # macro-averaged sensitivity
        'val_spec'    : [],    # macro-averaged specificity
    }
    best_f1    = -1.0
    best_epoch = 0
    no_improve = 0

    print(f"\n  {'Epoch':>5}  {'Train Loss':>11}  {'Val Loss':>9}  "
          f"{'Val F1':>8}  {'Sens':>7}  {'Spec':>7}  {'LR':>9}  {'Time':>6}")
    print(f"  {'─'*73}")

    for epoch in range(1, MAX_EPOCHS + 1):
        t0 = time.time()

        train_loss                        = train_one_epoch(model, train_loader,
                                                            optimiser, criterion, device)
        val_loss, val_f1, val_sens, val_spec = evaluate(model, val_loader,
                                                         criterion, device)
        scheduler.step(val_f1)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_f1'].append(val_f1)
        history['val_sens'].append(val_sens)
        history['val_spec'].append(val_spec)

        lr     = optimiser.param_groups[0]['lr']
        marker = " ★" if val_f1 > best_f1 else ""
        print(f"  {epoch:>5}  {train_loss:>11.4f}  {val_loss:>9.4f}  "
              f"{val_f1:>8.4f}  {val_sens:>7.4f}  {val_spec:>7.4f}  "
              f"{lr:>9.2e}  {time.time()-t0:>5.1f}s{marker}")

        if val_f1 > best_f1:
            best_f1    = val_f1
            best_epoch = epoch
            no_improve = 0
            torch.save(model.state_dict(), checkpoint_path)
        else:
            no_improve += 1

        if no_improve >= EARLY_STOP_PATIENCE:
            print(f"\n  ⏹  Early stopping at epoch {epoch}")
            break

    model.load_state_dict(torch.load(checkpoint_path, weights_only=True))

    history['best_epoch'] = best_epoch
    history['best_f1']    = best_f1
    history['best_sens']  = history['val_sens'][best_epoch - 1]
    history['best_spec']  = history['val_spec'][best_epoch - 1]

    print(f"\n  ✅ Best val macro-F1: {best_f1:.4f}  "
          f"Sens: {history['best_sens']:.4f}  "
          f"Spec: {history['best_spec']:.4f}  "
          f"@ epoch {best_epoch}")

    return history

In [75]:
assert MODEL   in MODEL_REGISTRY, \
    f"Unknown MODEL '{MODEL}'. Choose from: {list(MODEL_REGISTRY.keys())}"
assert LOSS_FN in LOSS_REGISTRY,  \
    f"Unknown LOSS_FN '{LOSS_FN}'. Choose from: {list(LOSS_REGISTRY.keys())}"

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
Path(CHECKPOINTS_DIR).mkdir(parents=True, exist_ok=True)
Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)

print(f"Device   : {device}")
print(f"Model    : {MODEL}")
print(f"Loss     : {LOSS_FN}")
print(f"Strategy : {STRATEGY}")

all_histories = []

for split in all_splits:
    fold_label = f"fold{split['fold']}" if split['fold'] else STRATEGY
    run_name   = f"{MODEL.lower().replace('-','')}_{LOSS_FN.lower()}_{fold_label}"

    print(f"\n{'═'*60}")
    print(f"  Run: {run_name}")
    print(f"  Train: {len(split['train'])} images  "
          f"Val: {len(split['val'])} images  "
          f"Test: {len(split['test'])} images")
    print(f"{'═'*60}")

    checkpoint_path = f"{CHECKPOINTS_DIR}/{run_name}.pt"

    train_loader, val_loader, class_weights = build_loaders(
        split['train'], split['val']
    )

    model     = MODEL_REGISTRY[MODEL]().to(device)
    criterion = LOSS_REGISTRY[LOSS_FN](class_weights.to(device))

    print(f"\n  Params : {sum(p.numel() for p in model.parameters()):,}")

    history = train(
        model           = model,
        train_loader    = train_loader,
        val_loader      = val_loader,
        criterion       = criterion,
        device          = device,
        checkpoint_path = checkpoint_path,
    )

    history['run_name']      = run_name
    history['test_patients'] = [p['id'] for p in split['test']]
    all_histories.append(history)

    np.save(f"{RESULTS_DIR}/{run_name}_history.npy", history)
    print(f"  History   → {RESULTS_DIR}/{run_name}_history.npy")
    print(f"  Checkpoint → {checkpoint_path}")

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"\n{'═'*60}")
print(f"  {'Run':<40} {'Best F1':>8}  {'Epoch':>6}")
print(f"  {'─'*55}")
for h in all_histories:
    print(f"  {h['run_name']:<40} {h['best_f1']:>8.4f}  {h['best_epoch']:>6}")

Device   : cuda
Model    : 1D-NN
Loss     : CE
Strategy : vp1

════════════════════════════════════════════════════════════
  Run: 1dnn_ce_vp1
  Train: 35 images  Val: 16 images  Test: 10 images
════════════════════════════════════════════════════════════

Class weights (from training set):
  Normal Tissue (NT)       :    193,476 px  →  0.0698
  Tumour Tissue (TT)       :     19,966 px  →  0.6760
  Blood Vessel (BV)        :     80,158 px  →  0.1684
  Background (BG)          :    157,332 px  →  0.0858

  Params : 17,055,748

  Epoch   Train Loss   Val Loss    Val F1     Sens     Spec         LR    Time
  ─────────────────────────────────────────────────────────────────────────
      1       0.4281     4.5308    0.6946   0.7982   0.9483   1.00e-03   60.3s ★
      2       0.3668     6.2174    0.7027   0.7880   0.9515   1.00e-03   60.5s ★
      3       0.3535    12.3111    0.6810   0.7838   0.9460   1.00e-03   60.4s
      4       0.3415    28.8192    0.6810   0.8019   0.9411   1.00e-03  